## LLM Document Summarization and Retrieval of Specific Attributes
Author: **Peeyush Sharma**; Feedback: **PSharma3@gmail.com**

This notebooks showcases the ability of LLMs to summarize legal documents based on a very specific listing of attributes requested by end user. Specific listing of data attributes allows LLMs to focus on the precise set of attributes user desires and avoid judging the important of attributes on its own. As an example, this notebook retrieves main loan *terms* of a credit agreement (SEC url: https://www.sec.gov/Archives/edgar/data/1701758/000121390018004741/fs12018ex10-1_thelovesac.htm) as:


- **Interest Rates:**
  - Base Rate Loans: Base Rate + Applicable Margin (1.00% to 1.25%)
  - LIBO Rate Loans: Adjusted LIBO Rate + Applicable Margin (2.00% to 2.25%)
  - Default Rate: Applicable rate + 2% per annum
  - Applicable margins vary based on Monthly Average Excess Availability

- **Fees:**
  - Commitment Fee: 0.375% per annum on unused commitments
  - Letter of Credit Fees: Applicable Margin for LIBO Rate Loans multiplied by daily stated amount
  - Fronting Fee: 0.125% per annum on face amount of Letters of Credit
  - Assignment fee: $3,500
  - Fee for Committed Increase: 0.50% of increase amount

- **Liens:**
  - First priority security interest in all Collateral
  - Various Permitted Encumbrances allowed


Read on for more details. Refer to the last cell for final output. The reference document being summarized here is available on SEC: https://www.sec.gov/Archives/edgar/data/1701758/000121390018004741/fs12018ex10-1_thelovesac.htm

In [1]:
import os
import re

import pypdf
import anthropic

In [2]:
# Basic setup/config items
DOC_DIR = "../documents/Credit"
FILE_NAME = "sec.gov_Archives_edgar_data_1701758_000121390018004741_fs12018ex10-1_thelovesac.htm.pdf"
# LLM_MODEL_NAME = "claude-3-5-sonnet-20241022"
LLM_MODEL_NAME = "claude-opus-4-20250514"

In [7]:
# Custom created list based on broad readings of source document: https://www.sec.gov/Archives/edgar/data/1701758/000121390018004741/fs12018ex10-1_thelovesac.htm
details_to_extract = [
    'Parties involved (lenders, agent, borrower)',
    'Parties details (address, business, description)',
    'Terms (interest charged, rates, fees, payments, liens)',
    'Loan details (credit limit, letter of credit, repayments, timeline)'
    'Payments (agent clawback, sharing of payments, settlement among lenders)',
    'Miscellaneous (litigation, insurance, taxes)'
]

In [4]:
# Function to read the PDF
def get_llm_text(pdf_file):
    reader = pypdf.PdfReader(pdf_file)
    text = "\n".join([page.extract_text() for page in reader.pages])

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)

    # Remove page numbers
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    return text

In [5]:
pdf = (os.path.join(DOC_DIR, FILE_NAME))
document_text = get_llm_text(pdf)
print(document_text[:50])

EX-10.1 5 fs12018ex10-1_thelovesac.htm WELLS FARGO


In [6]:
# Initialize the Anthropic client
client = anthropic.Anthropic()

def summarize_document(text, details_to_extract, model=LLM_MODEL_NAME, max_tokens=1000):
    # Format the details to extract to be placed within the prompt's context
    details_to_extract_str = '\n'.join(details_to_extract)

    # Prompt the model to summarize the loan agreement
    prompt = f"""Summarize the following credit agreement. Focus on these key aspects:

    {details_to_extract_str}

    Provide the summary in bullet points nested within the XML header for each section. For example:

    <parties involved>
    - Lender: [Name]
    // Add more details as needed
    </parties involved>

    If any information is not explicitly stated in the document, note it as "Not specified". Do not preamble.

    Loan agreement text:
    {text}
    """

    response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        system="You are a legal analyst specializing in credit law, known for highly accurate and detailed summaries of credit agreements.",
        messages=[
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": "Here is the summary of the credit agreement: <summary>"}
        ],
        stop_sequences=["</summary>"]
    )

    return response.content[0].text


loan_summary = summarize_document(document_text, details_to_extract)
print(loan_summary)



<parties involved>
- Lender: Wells Fargo Bank, National Association (as Agent, L/C Issuer, Swing Line Lender, Sole Lead Arranger, and Sole Bookrunner)
- Borrower: The Lovesac Company (Delaware corporation) - Lead Borrower
- Guarantor: SAC Acquisition LLC (Delaware limited liability company) - Parent
</parties involved>

<parties details>
- The Lovesac Company: Delaware corporation
- SAC Acquisition LLC: Delaware limited liability company, owns 100% of equity interests in The Lovesac Company
- Wells Fargo Bank, National Association: Acting in multiple capacities (Agent, Lender, L/C Issuer, etc.)
- All parties' specific addresses listed in Schedule 10.02 (not included in excerpt)
</parties details>

<terms>
Interest Rates:
- Base Rate Loans: Base Rate + Applicable Margin (1.00% to 1.25%)
- LIBO Rate Loans: Adjusted LIBO Rate + Applicable Margin (2.00% to 2.25%)
- Default Rate: Applicable rate + 2% per annum
- Applicable margins vary based on Monthly Average Excess Availability

Fees:
-

Feel Free to cross check the extracted data points against the source document available at SEC portal here:
https://www.sec.gov/Archives/edgar/data/1701758/000121390018004741/fs12018ex10-1_thelovesac.htm